In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

In [2]:
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()), Path.cwd())
sys.path.insert(0, str(ROOT))
from src.pipeline_paths import ARTIFACT_DIR, METRICS_DIR  # noqa: E402

In [3]:
forecast = pd.read_parquet(ARTIFACT_DIR / "forecasts" / "forecast.parquet")
comparison = json.loads((METRICS_DIR / "comparison.json").read_text(encoding="utf-8"))
chosen = next(r for r in comparison["candidates"] if r["model"] == comparison["selected"])
error = chosen["primary_mean"]

In [4]:
print("Model used:", comparison["selected"])
print("Period:", forecast["date"].min().date(), "to", forecast["date"].max().date())
print(f"Typical error (WAPE): {error:.0%} of actual demand, measured on past periods")

Model used: xgboost
Period: 2026-01-01 to 2026-01-28
Typical error (WAPE): 71% of actual demand, measured on past periods


In [5]:
by_store = forecast.groupby("store_key")["forecast"].sum().round(0)
by_category = forecast.groupby("category")["forecast"].sum().round(0)
print("Units per store:")
print(by_store.to_string())
print()
print("Units per category:")
print(by_category.to_string())
print()
print("All stores:", int(by_store.sum()), "units")

Units per store:
store_key
1    350.0
2    339.0
3    356.0

Units per category:
category
Apparel        261.0
Electronics    263.0
Grocery        255.0
Home           266.0

All stores: 1045 units


In [6]:
table = forecast.pivot_table(index="store_key", columns="category", values="forecast", aggfunc="sum").round(0)
table

category,Apparel,Electronics,Grocery,Home
store_key,,,,
1,88.0,91.0,79.0,92.0
2,87.0,83.0,86.0,83.0
3,86.0,90.0,90.0,91.0


In [7]:
by_day = forecast.groupby("date")["forecast"].sum().round(1)
print("Busiest days:")
print(by_day.nlargest(5).to_string())
print()
print("Slowest days:")
print(by_day.nsmallest(5).to_string())

Busiest days:
date
2026-01-18    39.400002
2026-01-23    39.000000
2026-01-21    38.700001
2026-01-17    38.599998
2026-01-22    38.599998

Slowest days:
date
2026-01-03    34.700001
2026-01-01    35.200001
2026-01-06    35.400002
2026-01-05    35.599998
2026-01-02    35.700001


In [8]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = forecast.assign(day=forecast["date"].dt.day_name()).groupby("day")["forecast"].mean()
weekday.reindex(order).round(2)

day
Monday       3.10
Tuesday      3.09
Wednesday    3.16
Thursday     3.08
Friday       3.11
Saturday     3.11
Sunday       3.12
Name: forecast, dtype: float32